In [54]:
import pandas as pd 
import numpy as np
import requests
from io import StringIO
from datetime import timedelta, datetime
#ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [55]:
def read_csv_from_github(url):
    """
    Reads a CSV file from a given GitHub URL and returns it as a pandas DataFrame.
    input:
        url = URL of the CSV file on GitHub
    output:
        df = DataFrame containing the CSV data
    """
    response = requests.get(url)
    if response.status_code == 200:
        csv_content = response.content.decode('utf-8')
        return pd.read_csv(StringIO(csv_content))
    else:
        print(f"Failed to fetch file. Status code: {response.status_code}, Message: {response.text}")
        return pd.DataFrame()
        
def loading_surveillance_eval(start_date, github_repo, github_directory, surveillance_file, state, horizon_to_start):
    """
    This function loads the surveillance data from a GitHub repository.
    input:
        start_date = start date for the data
        github_repo = GitHub repository name
        github_directory = directory in the GitHub repository
        surveillance_file = name of the surveillance file
        state = state to filter the data
        horizon_to_start = horizon to start the data
    output:
        df_surv_date_US = dataframe with the filtered surveillance data
    """
    file_url = f"https://raw.githubusercontent.com/{github_repo}/main/{github_directory}/{surveillance_file}"
    df_surv = read_csv_from_github(file_url)
    start_date = pd.to_datetime(start_date)
    df_surv['date'] = pd.to_datetime(df_surv['date'])
    if state == 'all':
        df_surv_date_US = df_surv[(df_surv.date >= start_date)]
    else:
        df_surv_date_US = df_surv[(df_surv.location == state) & (df_surv.date >= start_date)]
    df_surv_date_US = df_surv_date_US.rename(columns={"value": "hospitalizations"})
    df_surv_date_US = df_surv_date_US.sort_values(by='date')
    df_surv_date_US['horizon'] = np.arange(1, len(df_surv_date_US)+1)
    if state == 'all':
        df_surv_date_US['horizon'] = df_surv_date_US.groupby('location').cumcount() + 1
    if 'X' in df_surv_date_US.columns:
        df_surv_date_US.drop(columns=['X'], inplace=True)
    df_surv_date_US = df_surv_date_US[df_surv_date_US['horizon'] >= horizon_to_start]
    df_surv_date_US['date'] = pd.to_datetime(df_surv_date_US['date'])
    return df_surv_date_US

In [56]:
state = 'US'
github_repo = "cdcepi/FluSight-forecast-hub"
github_directory = "auxiliary-data/target-data-archive"
# Loading surveillance data for evaluation: last surveillance file
start_date = datetime(2024, 11, 16)  # Adjusted to the latest date in the surveillance data
surveillance_file = "target-hospital-admissions_2025-06-14.csv"
horizon_to_start = 3
df_surv = loading_surveillance_eval(start_date, github_repo, github_directory, surveillance_file, state, horizon_to_start)
df_surv = df_surv.iloc[:-2]
df_surv['horizon'] = df_surv['horizon'] + 13
df_surv

,date,location,location_name,hospitalizations,weekly_rate,horizon
3888,2024-11-30,US,US,4261.0,1.272264,16
4638,2024-12-07,US,US,6271.0,1.872416,17
3889,2024-12-14,US,US,9131.0,2.726364,18
6956,2024-12-21,US,US,15374.0,4.590420,19
5735,2024-12-28,US,US,27597.0,8.240004,20
820,2025-01-04,US,US,38611.0,11.528600,21
1438,2025-01-11,US,US,30662.0,9.155162,22
6957,2025-01-18,US,US,32908.0,9.825780,23
5736,2025-01-25,US,US,40513.0,12.096506,24
9317,2025-02-01,US,US,51050.0,15.242678,25


In [ ]:
loss_function = 'wmape'
season = "2024-2025"
df_original_ensemble = pd.read_csv(f'../../output_data/original_ensembles_bootstrap/original_scenarios_S2_bootstrap_US_{loss_function}_{season}_k_values.csv')
df_original_ensemble = df_original_ensemble[df_original_ensemble['scenario_id'] == 'Ens2']
df_original_ensemble

,quantile,value,horizon,scenario_id,n_bootstrapping
57960,0.010,13.038024,1,Ens2,0
57961,0.025,15.281052,1,Ens2,0
57962,0.050,28.997489,1,Ens2,0
57963,0.100,78.671926,1,Ens2,0
57964,0.150,87.448307,1,Ens2,0
...,...,...,...,...,...
67615,0.850,826.549149,42,Ens2,9
67616,0.900,1045.109598,42,Ens2,9
67617,0.950,3073.775000,42,Ens2,9
67618,0.975,3482.272500,42,Ens2,9


In [59]:
def import_ensemble_original(df, df_surv, date_ref):
    df_surv['date'] = pd.to_datetime(df_surv['date'])
    valid_horizons = df_surv['horizon'].unique()
    df = df[df['horizon'].isin(valid_horizons)]
    # Map each horizon to its corresponding date
    horizon_to_date = df_surv.set_index('horizon')['date'].to_dict()
    df['date'] = df['horizon'].map(horizon_to_date)
    # Filter to only include dates after or equal to date_only
    date_ref = pd.to_datetime(date_ref)
    df = df[df['date'] >= date_ref]
    # Merge in hospitalization targets
    df = df.merge(df_surv[['date', 'hospitalizations']].drop_duplicates(), on='date', how='left')
    return df

def import_ensemble2(file_path, surv_lookup, horizon_to_date):
    df = pd.read_csv(file_path, index_col=0)
    df['date'] = df['horizon'].map(horizon_to_date)
    df['date'] = pd.to_datetime(df['date'])
    return df.merge(surv_lookup, on='date', how='left')   

In [60]:
def get_perc_error(actual, sim, last=4): 
    return 100 * np.mean(np.abs(actual[-last:] - sim[-last:]) / actual[-last:])

def get_wmape(actual, sim) -> float:
    return np.sum(np.abs(actual - sim)) / np.sum(np.abs(actual))

def diff(a, b, norm=False): 
    if norm:
        if a != 0:
            return (a - b) / a
        else: 
            return 0
    else:
        return (a - b)

def interval_score(y, u, l, alpha, norm=False): 
    return -diff(l, u, norm=norm) + 2 / alpha * -diff(y, l, norm=norm) * (y < l) + 2 / alpha * diff(y, u, norm=norm) * (y > u)
    

def weighted_interval_score(y, u_s, l_s, m, alpha_ks, w0=1./2., norm=False):
    K = len(alpha_ks)
    wks = np.array(alpha_ks) / 2.
    return 1. / (K + 1./2.) * (w0 * np.abs(diff(y, m, norm=norm)) + np.dot(wks, [interval_score(y, u, l, a_k, norm=norm) for u, l, a_k in zip(u_s, l_s, alpha_ks)]))


def get_upper_bound(sim_stats, alpha, idx): 
    # get upper bound levels
    q2 = 1.0 - alpha / 2
    return sim_stats.loc[sim_stats["quantile"] == q2]["value"].values[idx]

def get_lower_bound(sim_stats, alpha, idx): 
    # get lower bound levels
    q1 = alpha / 2.
    return sim_stats.loc[sim_stats["quantile"] == q1]["value"].values[idx]


def get_aggregate_wis(realdata, 
                      sim_stats, 
                      alphas=[0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90], 
                      aggr_fun=None, 
                      norm=False):
    wis = []
    for i in range(len(realdata)):
        wis.append(weighted_interval_score(
                            y=realdata[i], 
                            u_s=np.array([get_upper_bound(sim_stats, alpha=alpha, idx=i) for alpha in alphas]), 
                            l_s=np.array([get_lower_bound(sim_stats, alpha=alpha, idx=i) for alpha in alphas]), 
                            m=sim_stats.loc[sim_stats["quantile"] == 0.5]["value"].values[i],
                            alpha_ks=alphas, 
                            norm=norm))
    if aggr_fun != None:
        return aggr_fun(wis)
    else:
        return wis
        
def compute_dict_WIS_AE(df, alphas):
    if 'quantiles' in df.columns:
        df = df.rename(columns={"quantiles": "quantile"})
    realdata = df[df['quantile'] == 0.5]['hospitalizations'].values
    sim_stats = df.drop(columns=['date', 'hospitalizations', 'horizon'])
    wmape = get_wmape(realdata, sim_stats.loc[sim_stats["quantile"] == 0.5]["value"].values)
    wis_list = get_aggregate_wis(realdata, 
                      sim_stats, 
                      alphas=alphas, 
                      aggr_fun=None, 
                      norm=False)
    wis_mean = np.mean(wis_list)
    return wis_list, wis_mean, wmape

In [ ]:
adaptive_ensemble2_path = "../../output_data/adaptive_ensemble_bootstrap"
horizon_to_date = df_surv.set_index('horizon')['date'].to_dict()
surv_lookup = df_surv[['date', 'hospitalizations']].drop_duplicates()
k_values = [0.05, 0.15, 0.25, 0.50, 0.75]
alphas=[0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

dict_k_wis_rounds_nbootstrap = {}
dict_k_mae_rounds_nbootstrap = {}

for n_bootstrap in df_original_ensemble.n_bootstrapping.unique():
    print('N bootstrap: ', n_bootstrap)
    dict_k_wis_rounds = {}
    dict_k_mae_rounds = {}
    for k in k_values:
        dict_wis_rounds = {}
        dict_mae_rounds = {}
        unique_dates = df_surv['date'].sort_values().unique()[:-1]
        for date in unique_dates:
            # Adaptive Ensemble2
            print(date)
            full_path_adaptive_ensemble2 = f"{adaptive_ensemble2_path}/{date.strftime('%Y-%m-%d')}_{k}_{loss_function}_{n_bootstrap}.csv"
            print(full_path_adaptive_ensemble2)
            df_adaptive_ensemble2 = import_ensemble2(full_path_adaptive_ensemble2, surv_lookup, horizon_to_date)
            print(df_adaptive_ensemble2)
            wis_list_adens2, wis_mean_adens2, wmape_adens2 = compute_dict_WIS_AE(df_adaptive_ensemble2, alphas)
            
            # Original ensemble
            df_original_ensemble_n = df_original_ensemble[df_original_ensemble['n_bootstrapping'] == n_bootstrap]
            df_original_ensemble2 = import_ensemble_original(df_original_ensemble_n, df_surv, date)
            wis_list_original_ens2, wis_mean_original_ens2, wmape_original_ens2 = compute_dict_WIS_AE(df_original_ensemble2, alphas)
            # Store WIS and AE results
            dict_wis_rounds[date] = [wis_mean_adens2, wis_mean_original_ens2]
            dict_mae_rounds[date] = [wmape_adens2, wmape_original_ens2]
        k_perc = int(k * 100)
        dict_k_wis_rounds[k_perc] = dict_wis_rounds
        dict_k_mae_rounds[k_perc] = dict_mae_rounds
    dict_k_wis_rounds_nbootstrap[n_bootstrap] = dict_k_wis_rounds
    dict_k_mae_rounds_nbootstrap[n_bootstrap] = dict_k_mae_rounds
    

N bootstrap:  0
2024-11-30 00:00:00
../output_data/review_adaptive_ensemble_bootstrap/2024-11-30_0.05_wmape_0.csv
     quantile        value  horizon       date  hospitalizations
0       0.010  2403.516000       16 2024-11-30            4261.0
1       0.025  2898.036567       16 2024-11-30            4261.0
2       0.050  3250.024217       16 2024-11-30            4261.0
3       0.100  3753.946926       16 2024-11-30            4261.0
4       0.150  4030.485051       16 2024-11-30            4261.0
..        ...          ...      ...        ...               ...
616     0.850   867.380000       42 2025-05-31            1740.0
617     0.900  3885.240000       42 2025-05-31            1740.0
618     0.950  3957.900000       42 2025-05-31            1740.0
619     0.975  4164.415000       42 2025-05-31            1740.0
620     0.990  4254.234000       42 2025-05-31            1740.0

[621 rows x 5 columns]
2024-12-07 00:00:00
../output_data/review_adaptive_ensemble_bootstrap/2024-12-07_0

In [62]:
dict_k_wis_rounds_nbootstrap

{0: {5: {Timestamp('2024-11-30 00:00:00'): [7893.633820732624,
    8059.021097013888],
   Timestamp('2024-12-07 00:00:00'): [7993.346942886773, 8217.24843648712],
   Timestamp('2024-12-14 00:00:00'): [9138.842960966262, 8370.062533978758],
   Timestamp('2024-12-21 00:00:00'): [9365.900175143812, 8533.872453316377],
   Timestamp('2024-12-28 00:00:00'): [9995.0691794949, 8743.795070724873],
   Timestamp('2025-01-04 00:00:00'): [9225.46718197008, 8938.832174999454],
   Timestamp('2025-01-11 00:00:00'): [8764.519571641575, 8847.9558052144],
   Timestamp('2025-01-18 00:00:00'): [9071.698862567815, 8930.707549576666],
   Timestamp('2025-01-25 00:00:00'): [9075.851162754609, 8915.688888232493],
   Timestamp('2025-02-01 00:00:00'): [8573.421798567819, 8575.64623506543],
   Timestamp('2025-02-08 00:00:00'): [7291.6625394546545, 7667.202009497036],
   Timestamp('2025-02-15 00:00:00'): [5872.728738887064, 6356.631807832168],
   Timestamp('2025-02-22 00:00:00'): [4777.546193478656, 5222.8380802528

In [ ]:
# Create df_wis dataframe to store WIS results
rows = []
for n, n_grouped_data in dict_k_wis_rounds_nbootstrap.items():
    for k, group_data in n_grouped_data.items():
        for timestamp, values in group_data.items():
            rows.append({
                'n_bootstrap': n,
                'k_perc': k,
                'week': timestamp,
                'wis_adaptive_ensemble2': values[0],
                'wis_original_ensemble2': values[1],
                'wis_rel_original2': values[0] / values[1],
            })
df_wis = pd.DataFrame(rows)
df_wis
df_wis.to_csv(f"../../output_data/performance_evaluation_bootstrap/wis_performance_{loss_function}.csv", index=False)

In [ ]:
# Create df_mae dataframe to store MAE results
rows = []
for n, n_grouped_data in dict_k_mae_rounds_nbootstrap.items():
    for k, group_data in n_grouped_data.items():
        for timestamp, values in group_data.items():
            rows.append({
                'n_bootstrap': n,
                'k_perc': k,
                'week': timestamp,
                'mae_adaptive_ensemble2': values[0],
                'mae_original_ensemble2': values[1],
                'mae_rel_original2': values[0] / values[1]
            })
df_mae = pd.DataFrame(rows)
df_mae.to_csv(f"../../output_data/performance_evaluation_bootstrap/mae_performance_{loss_function}.csv", index=False)
df_mae

,n_bootstrap,k_perc,week,mae_adaptive_ensemble2,mae_original_ensemble2,mae_rel_original2
0,0,5,2024-11-30,0.585630,0.638259,0.917542
1,0,5,2024-12-07,0.587728,0.633404,0.927888
2,0,5,2024-12-14,0.607131,0.630741,0.962567
3,0,5,2024-12-21,0.609220,0.632029,0.963912
4,0,5,2024-12-28,0.625568,0.648225,0.965048
...,...,...,...,...,...,...
1295,9,75,2025-04-26,0.747595,0.751716,0.994518
1296,9,75,2025-05-03,0.753228,0.754769,0.997959
1297,9,75,2025-05-10,0.764341,0.764355,0.999982
1298,9,75,2025-05-17,0.783871,0.781760,1.002700
